# 8. Deep Learning Architectures

## 8.1 CNN

### 8.1.1 Introduction

For images, if we simply apply fully connected networks, it will produce massive trainable parameters. Thus, CNN is introduced. By using convolutional filters and spatial structure of images, it can reduce the amount of parameters.

There are three important characteristics about CNN:

1. Local connectivity: one output element only depends on certain areas of the input;
2. Weight sharing: the same filter uses the same parameters throughout different spatial locations;
3. Feature hierarchy: multiple layers of CNN can build up different hierarchy of features, like Edges - Textures - Object.

### 8.1.2 Convolution Calculation

In deep-learning frameworks, convolution most often means Cross-Correlation, that is to do element-wise multiplication and sum on kernels and specific areas. For Example:


In [ ]:
import numpy as np

X = np.array(
    [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 9],
    ]
)

K = np.array(
    [
        [1, 0],
        [0, -1],
    ]
)

When using Stride=1, Padding=0, the output of the upper-left corner will be:

$$
1(1) + 2(0) + 4(0) + 5(-1) = -4
$$

For a 2D Convolution:

$$
H_{out} = \lfloor \frac{H+2P-K_h}{S} \rfloor + 1
$$

$$
W_{out} = \lfloor \frac{W+2P-K_w}{S} \rfloor +1
$$

in which:

- $H$, $W$ is the height and width of input;
- $K_h$, $K_w$ is the height and width of kernel;
- $P$ is the padding of each side;
- $S$ is the stride.


### 8.1.3 Multiple Filters & Channels

In reality, the input often contains multiple channels:


In [ ]:
import tensorflow as tf

tf.keras.layers.Conv2D(
    filters=16,
    kernel_size=(3, 3),
    padding="same",
    activation="relu",
)

# If the input is (32,32,3), the output will be (32,32,16)

Notice that the number of output channels is determined by the number of `filters`, instead of `kernel_size`.


### 8.1.4 Pooling Layers

Pooling is used in spatial downsampling for feature maps. There are several types of pooling:

- Max Pooling: Select the maximum value of the cell;
- Average Pooling: Calculate the mean value of the cell;
- Global Average Pooling: Calculate the mean value of the whole spatial map in one channel.

Meanwhile, no parameter is trained in pooling.


### 8.1.5 LeNet & ResNet

LeNet is a classic CNN structure, which has a fixed order of layers:

$$
Convolution \rightarrow Pooling \rightarrow Convolution \rightarrow Pooling \rightarrow Convolution \rightarrow Fullyconnected \rightarrow Output
$$

While deepening the networks, problems may appear in the model training process, such as Optimization Difficulties, and Gradient Propagation. ResNet introduces Residual Connection (Skip Connection):

$$
y = f(x) + x
$$


### 8.1.6 Keras CNN Implementation

Different from the dense-only network before, we keep the 2D spatial structure of images this time:


In [ ]:
import numpy as np
import tensorflow as tf

# Load data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize and add channel dimension
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(X_train.shape)  # (60000, 28, 28, 1)

# Build CNN
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(28, 28, 1),
        ),  # shape = (28,28,1)
        tf.keras.layers.Conv2D(
            16,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
        ),  # shape = (28,28,16)
        tf.keras.layers.MaxPooling2D(
            pool_size=(2, 2),
        ),  # shape = (14,14,16)
        tf.keras.layers.Conv2D(
            32,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
        ),  # shape = (14,14,32)
        tf.keras.layers.MaxPooling2D(
            pool_size=(2, 2),
        ),  # shape = (7,7,32)
        tf.keras.layers.Flatten(),  # shape = (1568,)
        tf.keras.layers.Dense(
            64,
            activation="relu",
        ),  # shape = (64,)
        tf.keras.layers.Dense(
            10,
            activation="softmax",
        ),  # shape = (10,)
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.1,
)

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0,
)

print("Test Accuracy:", test_accuracy)

in which the shape of an image is defined as:

$$
(Height, Width, Channels) = (H,W,C)
$$

Channels is the number of features of certain spatial location. For example, in a B-W image, `c=1`; In an RGB image, `c=3`. However, the number of channels is not restricted. In CNN, it can also represent the number of features.

Furtheremore, `padding="same"` auto sets the padding to ensure the shape of the output is the same as input.


### 8.1.7 Trainable Parameters

The Parameter Count of a Conv2D layer is defined as:

$$
(K_hK_wC_{in} + 1) \times C_{out}
$$


## 8.2 RNN, LSTM, GRU

### 8.2.1 Recurrent Neural Networks (RNN)

Recurrent Neural Networks (RNN) is good at dealing with sequential data like natrual language, while CNN is good at extracting spatial patterns.

The difference is that: the calculation in the current time step can use the Hidden State from the previous steps.

#### 8.2.1.1 Core Formula of RNN

The core formula of RNN is:

$$
h_t = \tanh (W_xx_t + W_hh_{t-1} + b)
$$

in which $x$ is input, $h$ is hidden state, $W$ is weight.

#### 8.2.1.2 Backpropagation Through Time (BPTT)

While training RNN, it will calculate the gradients backthrough the computational graph. It may cause:

- Vanishing Gradients: Gradients gradually closes to 0, making it hard to learn long-term dependencies;
- Exploding Gradients: Gradients are too large to stablize the training.

### 8.2.2 Long Short-Term Memory (LSTM)

LSTM is a special RNN Architecture with gating mechanisms, so that networks can better learn long-term dependencies.

Three Important Gates is set in LSTM:

- Forget Gate: determines hwo much information from old cell state will be kept;
- Input Gate: determines how much new information is written into cell state;
- Output Gate: determines the output from cell state to hidden state.

We can see that LSTM has two different states:

- $c_t$: Cell State, used to pass down long-term information;
- $h_t$: Hidden State, the output of the current time step.

### 8.2.3 Gated Recurrent Unit (GRU)

We can view GRU as another kind of gated RNN, but is simpler than LSTM. It only uses two gates:

- Update Gate: Controls the level of keeping previous state and applying new candidate state;
- Reset Gate: Controls how much previous hiddden state information is used to create new candidate states.

### 8.2.4 Keras Implementation

#### 8.2.4.1 Input Shape of RNN

RNN often receives a 3D Tensor:

$$
X.shape = (batch, timesteps, features)
$$

#### 8.2.4.2 SimpleRNN, LSTM and GRU


In [ ]:
import tensorflow as tf

rnn_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(20, 8)),
        tf.keras.layers.SimpleRNN(32),
        tf.keras.layers.Dense(1),
    ]
)

lstm_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(20, 8)),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dense(1),
    ]
)

gru_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(20, 8)),
        tf.keras.layers.GRU(32),
        tf.keras.layers.Dense(1),
    ]
)

An important API in RNN is `return_sequences`:

- `return_sequences=False`: only returns the output of the last time step, whose shape is (batch, units);
- `return_sequences=True`: returns the outputs of all time steps, whose shape is (batch, timesteps, units).


In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(20, 8),
        ),  # shape = (None,20,8)
        tf.keras.layers.LSTM(
            32,
            return_sequences=True,
        ),  # shape = (None,20,32)
        tf.keras.layers.GRU(
            16,
            return_sequences=False,
        ),  # shape = (None,16)
        tf.keras.layers.Dense(
            3,
            activation="softmax",
        ),  # shape = (None,3)
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 8.3 Transformer & Self-Attention

### 8.3.1 From RNN to Transformer

The character of RNN is to process the sequence in order of time, and pass down information by Hidden States. Transformer mainly uses the Attention Mechanisms to build up direct association between tokens.

### 8.3.2 Query, Key, Value

The core of Sel-Attention is the combination of three vectors:

- Query (Q): what information the current token wants?
- Key (K): what information should be given to each token?
- Value (V): the combined information produced from each Q-K set.

They obey the following formula:

$$
Attention(Q,K,V) = \text{softmax} (\frac{QK^T}{\sqrt{d_k}}) V
$$

### 8.3.3 Matrix Shapes

Suppose a Sequence with 4 tokens, K/Q dimension are 8, V dimension is 6, then:


In [ ]:
import numpy as np

Q = np.zeros((4, 8))
K = np.zeros((4, 8))
V = np.zeros((4, 6))
weights = np.zeros((4, 4))

print(Q.shape == (4, 8))
print(K.shape == (4, 8))
print(V.shape == (4, 6))

scores = Q @ K.T  # (4, 4)
output = weights @ V  # (4, 6)

### 8.3.4 Manual Implementation of Self-Attention

Let's use a dataset with 2 tokens as an example:


In [ ]:
import numpy as np

Q = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
    ]
)

K = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
    ]
)

V = np.array(
    [
        [2.0, 0.0],
        [0.0, 4.0],
    ]
)

# Step 1: Compute scaled attention scores
d_k = K.shape[-1]
scores = Q @ K.T / np.sqrt(d_k)

# Step 2: Stable row-wise softmax
shifted = scores - np.max(
    scores,
    axis=-1,
    keepdims=True,  # keepdims keeps the dimension, allowing following bradcasting.
)
exp_scores = np.exp(shifted)
weights = exp_scores / np.sum(
    exp_scores,
    axis=-1,
    keepdims=True,
)

# Step 3: Weighted sum of values
output = weights @ V

print("Scores:\n", scores)
print("Attention Weights:\n", weights)
print("Output:\n", output)

### 8.3.5 Multi-Head Attention

Standard Self-Attention only calculates one set of QKV, while multi-head attention uses multiple learned projects, with different attention head to produce different results.


### 8.3.6 Positional Encoding

Self-Attention doesn't natrually have order, thus it needs a kind of positional information.

Traditional Transformers uses Sinusodial Positional Encoding:

$$
PE(pos,2i) = \sin (\frac{pos}{1000^{2i/d}})
$$

$$
PE(pos,2i+1) = \cos (\frac{pos}{1000^{2i/d}})
$$


### 8.3.7 Transformer Architectures

There are basically three types of Transformers:

- Encoder-only: used for bidirectional context representation;
- Decoder-only: casual self-attention, autoregressive generation, language models;
- Encoder-Decoder: used in translation.

Furthermore, decoder uses Casual Mask to prevent leakage of future tokens.


## 8.4 Generative AI & Multimodal Learning

### 8.4.1 Generative Adversarial Networks (GAN)

The core concept of GAN is Adversarial Training. It contains two neural networks:

- Generator (G): Try to generate samples that match the data distribution;
- Discriminator (D): Try to distinguish real samples and generated samples.

Thus, thr train works in a kind of iteration:

1. Generator produces Fake samples from random noise;
2. Discriminator accept both real and fake samples;
3. Discriminator try to learn how to distinguish real and fake;
4. Generator learn how to produce more hazy samples that are harder to distinguish;
5. Repeat the same process.

A common pitfall of GAN is Mode Collapse, in which case the model is only able to produce specific kind of samples.

### 8.4.2 Diffusion Models

A classic Diffusion Models works in the following steps:

- Forward Diffusion: add noise to the data step by step;
- Reverse Denoising: learn how to cut off the noise and produce new samples that is close to the original data.

### 8.4.3 Multimodal AI

Multimodal AI is a kind of AI that can deal with a mixed categories of data, including text, images, audio, video, etc.
